# Chapter 19 — $L^p$ Spaces

Companion notebook to Chapter 19 of *Advanced Calculus for Machine Learning*.
Every numerical claim of the chapter is recomputed here with PyTorch and
checked with an assertion.

Most computations are exact (fractions) or deterministic (finite measure
spaces, midpoint rule on a fine grid). Random numbers are used to generate
test vectors and, in three cells that say so, samples.

In [1]:
import math
import warnings
from fractions import Fraction

import torch

warnings.filterwarnings("ignore", category=FutureWarning)
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
print(torch.__version__)
T = torch.tensor


def midpoints(a, b, n):
    # Midpoints of n equal cells of [a, b] and the cell length.
    h = (b - a) / n
    return a + h * (torch.arange(n) + 0.5), h


def integral(f, a, b, n=2_000_000):
    x, h = midpoints(a, b, n)
    return (f(x) * h).sum()


def norm(f, w, p):
    # ||f||_p on a finite measure space with weights w.
    if p == math.inf:
        return f[w > 0].abs().max()
    return ((f.abs()**p * w).sum())**(1 / p)

2.15.0+cu132


## 1. Jensen's inequality (Theorem 19.1, Exercise 19.6)

A random variable with finitely many values, and four convex functions.
For the strictly convex ones equality holds only for a constant.

In [2]:
prob = torch.rand(8)
prob = prob / prob.sum()                              # a probability on 8 points
Z = 0.2 + 3 * torch.rand(8)                           # positive values
E = lambda v: (v * prob).sum()

convex = {"exp": torch.exp, "-ln": lambda t: -torch.log(t), "t^2": lambda t: t**2,
          "t ln t": lambda t: t * torch.log(t)}
for name, phi in convex.items():
    lhs, rhs = phi(E(Z)).item(), E(phi(Z)).item()
    const = torch.full_like(Z, 1.7)
    print(f"{name:7s} phi(E Z) = {lhs:+.6f} <= E phi(Z) = {rhs:+.6f}")
    assert lhs < rhs
    assert abs(phi(E(const)) - E(phi(const))) < 1e-12        # equality for a constant

# Exercise 19.6
assert E(Z.abs())**2 < E(Z**2)                               # (1)
assert torch.exp(E(torch.log(Z))) < E(Z)                     # (2)
assert E(Z) < torch.log(E(torch.exp(Z)))                     # (3)
z = T([1.0, 4.0, 2.0, 8.0, 0.5])
geometric, arithmetic = z.prod()**(1 / 5), z.mean()
print(f"geometric mean {geometric:.4f} <= arithmetic mean {arithmetic:.4f}")
assert geometric < arithmetic

exp     phi(E Z) = +4.282239 <= E phi(Z) = +4.775162
-ln     phi(E Z) = -0.374646 <= E phi(Z) = -0.311782
t^2     phi(E Z) = +2.115500 <= E phi(Z) = +2.345285
t ln t  phi(E Z) = +0.544913 <= E phi(Z) = +0.628290
geometric mean 2.0000 <= arithmetic mean 3.1000


## 2. Hölder and Minkowski (Theorems 19.3 and 19.4, Exercise 19.3)

On a finite measure space with random weights, for several conjugate pairs.
The inequality of Hölder is an equality for
$g = \operatorname{sign}(f)|f|^{q-1}/\|f\|_q^{q-1}$: the dual norm of
$\|\cdot\|_p$ is $\|\cdot\|_q$.

In [3]:
n = 50
w = 0.1 + torch.rand(n)                               # the measure
for p in (1.0, 1.5, 2.0, 3.0, 7.0, math.inf):
    q = math.inf if p == 1.0 else (1.0 if p == math.inf else p / (p - 1))
    worst_h, worst_m = 0.0, 0.0
    for _ in range(200):
        f, g = torch.randn(n), torch.randn(n)
        holder = ((f * g).abs() * w).sum() / (norm(f, w, p) * norm(g, w, q))
        minkowski = norm(f + g, w, p) / (norm(f, w, p) + norm(g, w, p))
        worst_h, worst_m = max(worst_h, holder.item()), max(worst_m, minkowski.item())
    print(f"p = {p:4}, q = {q:6.3f}:  max of Hoelder ratio = {worst_h:.4f}   "
          f"max of Minkowski ratio = {worst_m:.4f}")
    assert worst_h <= 1 + 1e-12 and worst_m <= 1 + 1e-12

# Young's inequality on a grid
a, b = torch.meshgrid(torch.linspace(0, 5, 201), torch.linspace(0, 5, 201), indexing="ij")
for p in (1.5, 2.0, 4.0):
    q = p / (p - 1)
    assert (a * b <= a**p / p + b**q / q + 1e-12).all()

# Equality case (Exercise 19.3)
for p in (1.5, 2.0, 3.0):
    q = p / (p - 1)
    f = torch.randn(n)
    g = torch.sign(f) * f.abs()**(q - 1) / norm(f, w, q)**(q - 1)
    print(f"p = {p}:  ||g||_p = {norm(g, w, p):.10f}   int f g = {(f * g * w).sum():.8f}"
          f"   ||f||_q = {norm(f, w, q):.8f}")
    assert abs(norm(g, w, p) - 1) < 1e-10 and abs((f * g * w).sum() - norm(f, w, q)) < 1e-10

p =  1.0, q =    inf:  max of Hoelder ratio = 0.4773   max of Minkowski ratio = 0.8807
p =  1.5, q =  3.000:  max of Hoelder ratio = 0.7796   max of Minkowski ratio = 0.8617
p =  2.0, q =  2.000:  max of Hoelder ratio = 0.8158   max of Minkowski ratio = 0.8354
p =  3.0, q =  1.500:  max of Hoelder ratio = 0.7850   max of Minkowski ratio = 0.8527
p =  7.0, q =  1.167:  max of Hoelder ratio = 0.6665   max of Minkowski ratio = 0.8933
p =  inf, q =  1.000:  max of Hoelder ratio = 0.5055   max of Minkowski ratio = 1.0000
p = 1.5:  ||g||_p = 1.0000000000   int f g = 3.54903404   ||f||_q = 3.54903404
p = 2.0:  ||g||_p = 1.0000000000   int f g = 6.50753288   ||f||_q = 6.50753288
p = 3.0:  ||g||_p = 1.0000000000   int f g = 7.99300351   ||f||_q = 7.99300351


## 3. Comparing exponents (Proposition 19.5, the pitfall after it, Exercises 19.1 and 19.2)

On a probability space $\|f\|_p$ increases with $p$; for the counting
measure it decreases. For $f(x) = x$ on $[0, 1]$,
$\|f\|_p = (p + 1)^{-1/p}\to1 = \|f\|_\infty$.

In [4]:
ps = [1.0, 1.5, 2.0, 3.0, 6.0, 20.0, math.inf]
f = torch.randn(n)
prob_w = w / w.sum()
counting = torch.ones(n)
on_prob = [norm(f, prob_w, p).item() for p in ps]
on_count = [norm(f, counting, p).item() for p in ps]
print("probability:", [f"{v:.4f}" for v in on_prob])
print("counting:   ", [f"{v:.4f}" for v in on_count])
assert all(a <= b + 1e-12 for a, b in zip(on_prob[:-1], on_prob[1:]))
assert all(a >= b - 1e-12 for a, b in zip(on_count[:-1], on_count[1:]))

# a general finite measure: ||f||_p <= mu(X)^(1/p - 1/r) ||f||_r
for p, r in [(1.0, 2.0), (2.0, 5.0), (1.5, math.inf)]:
    bound = w.sum()**(1 / p - (0 if r == math.inf else 1 / r)) * norm(f, w, r)
    assert norm(f, w, p) <= bound + 1e-12

for p in (1, 2, 10, 100, 1000):
    val = integral(lambda x: x**p, 0.0, 1.0).item()**(1 / p)
    print(f"p = {p:5d}:  ||x||_p = {val:.6f}   (p+1)^(-1/p) = {(p + 1)**(-1 / p):.6f}")
    assert abs(val - (p + 1)**(-1 / p)) < 1e-6

probability: ['0.6767', '0.7729', '0.8565', '0.9961', '1.3061', '1.9688', '2.4791']
counting:    ['35.5980', '10.9960', '6.3467', '3.8612', '2.6721', '2.4796', '2.4791']
p =     1:  ||x||_p = 0.500000   (p+1)^(-1/p) = 0.500000
p =     2:  ||x||_p = 0.577350   (p+1)^(-1/p) = 0.577350


p =    10:  ||x||_p = 0.786793   (p+1)^(-1/p) = 0.786793


p =   100:  ||x||_p = 0.954898   (p+1)^(-1/p) = 0.954898
p =  1000:  ||x||_p = 0.993115   (p+1)^(-1/p) = 0.993115


Exercise 19.1: $f = x^{-2/5}$ on $(0, 1)$ and $x^{-3/4}$ on $(1, \infty)$.
The integrals of $|f|^p$ over $(\epsilon, 1)$ and $(1, R)$ are computed in
closed form; they stay bounded as $\epsilon\to0$, $R\to\infty$ only for
$p = 2$, where the limit is $5 + 2 = 7$.

In [5]:
def power_integral(exponent, lo, hi):
    # int_lo^hi x^(-exponent) dx
    if abs(exponent - 1) < 1e-12:
        return math.log(hi / lo)
    return (hi**(1 - exponent) - lo**(1 - exponent)) / (1 - exponent)


for p in (1, 2, 3):
    row = []
    for scale in (1e3, 1e10, 1e40):
        head = power_integral(0.4 * p, 1 / scale, 1.0)
        tail = power_integral(0.75 * p, 1.0, scale)
        row.append(head + tail)
    print(f"p = {p}:  int |f|^p over (1/s, s) for s = 1e3, 1e10, 1e40:  "
          + "  ".join(f"{v:12.4g}" for v in row))
    if p == 2:
        assert abs(row[-1] - 7) < 1e-3
    else:
        assert row[-1] > 10 * row[0]

p = 1:  int |f|^p over (1/s, s) for s = 1e3, 1e10, 1e40:         20.13          1263         4e+10
p = 2:  int |f|^p over (1/s, s) for s = 1e3, 1e10, 1e40:         5.681          6.95             7
p = 3:  int |f|^p over (1/s, s) for s = 1e3, 1e10, 1e40:         15.71         495.8         5e+08


## 4. Convergence in $L^p$ and at points (Theorem 19.7, the pitfall after it, Figure 19.1, Exercise 19.4)

The ramps of Example 4.21 form a Cauchy sequence of continuous functions
for $\|\cdot\|_1$ whose limit, a step function, exists in $L^1$.

In [6]:
x, h = midpoints(0.0, 1.0, 1_000_000)
step = (x >= 0.5).double()
ramp = lambda k: ((x - 0.5) * k).clamp(0, 1)              # 0 up to 1/2, 1 after 1/2 + 1/k
for k in (2, 10, 100, 1000):
    d = ((ramp(k) - step).abs() * h).sum().item()
    d2 = ((ramp(k) - ramp(2 * k)).abs() * h).sum().item()
    print(f"k = {k:5d}:  ||f_k - step||_1 = {d:.6f} = 1/(2k)   ||f_k - f_2k||_1 = {d2:.6f}")
    assert abs(d - 1 / (2 * k)) < 1e-6

k =     2:  ||f_k - step||_1 = 0.250000 = 1/(2k)   ||f_k - f_2k||_1 = 0.125000
k =    10:  ||f_k - step||_1 = 0.050000 = 1/(2k)   ||f_k - f_2k||_1 = 0.025000
k =   100:  ||f_k - step||_1 = 0.005000 = 1/(2k)   ||f_k - f_2k||_1 = 0.002500
k =  1000:  ||f_k - step||_1 = 0.000500 = 1/(2k)   ||f_k - f_2k||_1 = 0.000250


The sliding indicators: $f_k$ is the indicator of
$[j2^{-n}, (j+1)2^{-n}]$ for $k = 2^n + j$. The norms tend to $0$, but at
any point the values $0$ and $1$ both come back in every generation.

In [7]:
def typewriter(k, pts):
    n = int(math.floor(math.log2(k)))
    j = k - 2**n
    return ((pts >= j / 2**n) & (pts <= (j + 1) / 2**n)).double()


pts = T([0.0, 0.1, 1 / 3, 0.5, math.sqrt(0.5), 1.0])
for p in (1, 2, 4):
    norms = [integral(lambda t: typewriter(k, t)**p, 0.0, 1.0, 1 << 16).item()**(1 / p)
             for k in (1, 2, 5, 37, 300)]
    exact = [2**(-math.floor(math.log2(k)) / p) for k in (1, 2, 5, 37, 300)]
    assert all(abs(a - b) < 1e-3 for a, b in zip(norms, exact))
    print(f"p = {p}:  ||f_k||_p for k = 1, 2, 5, 37, 300:", [f"{v:.4f}" for v in exact])

for gen in range(2, 12):
    vals = torch.stack([typewriter(k, pts) for k in range(2**gen, 2**(gen + 1))])
    assert (vals.max(dim=0).values == 1).all() and (vals.min(dim=0).values == 0).all()
print("generations 2..11: at each of the six points some f_k is 1 and some f_k is 0")

sub = torch.stack([typewriter(2**nn, pts) for nn in range(1, 30)])   # indicators of [0, 2^-n]
print("subsequence 1_[0, 2^-n] at the six points, n = 29:", sub[-1].tolist())
assert (sub[-1][1:] == 0).all() and sub[-1][0] == 1

p = 1:  ||f_k||_p for k = 1, 2, 5, 37, 300: ['1.0000', '0.5000', '0.2500', '0.0312', '0.0039']
p = 2:  ||f_k||_p for k = 1, 2, 5, 37, 300: ['1.0000', '0.7071', '0.5000', '0.1768', '0.0625']
p = 4:  ||f_k||_p for k = 1, 2, 5, 37, 300: ['1.0000', '0.8409', '0.7071', '0.4204', '0.2500']


generations 2..11: at each of the six points some f_k is 1 and some f_k is 0
subsequence 1_[0, 2^-n] at the six points, n = 29: [1.0, 0.0, 0.0, 0.0, 0.0, 0.0]


## 5. Dense subspaces (Theorem 19.8)

A disc approximated by unions of dyadic squares: the measure of the
symmetric difference tends to $0$. And the indicator of $[0, 1)$
approximated in $L^p$ by the continuous trapezoids of the proof, which are
$1$ on $[0, 1 - \delta]$ and $0$ outside $(-\delta, 1)$: the $p$-th power
of the distance is $2\delta/(p + 1)$.

In [8]:
fine = 2**11                                           # a fine grid on [-1, 1)^2 to measure sets
g, hf = midpoints(-1.0, 1.0, fine)
X, Y = torch.meshgrid(g, g, indexing="ij")
disc = (X**2 + Y**2 < 0.49)

previous = math.inf
for j in range(1, 8):
    side = 2 * 2**-j                                   # dyadic squares of this side
    cx = (torch.floor((X + 1) / side) + 0.5) * side - 1    # centre of the square containing (X, Y)
    cy = (torch.floor((Y + 1) / side) + 0.5) * side - 1
    union = (cx**2 + cy**2 < 0.49)                     # squares whose centre is in the disc
    sym_diff = (disc ^ union).double().sum().item() * hf * hf
    print(f"squares of side {side:.5f}:  measure of the symmetric difference = {sym_diff:.5f}")
    assert sym_diff < previous
    previous = sym_diff
assert previous < 0.03

x, h = midpoints(-1.0, 2.0, 3_000_000)
box = ((x >= 0) & (x < 1)).double()
for p in (1, 2, 5):
    for delta in (0.2, 0.02):
        trapezoid = torch.minimum(((x + delta) / delta).clamp(0, 1), ((1 - x) / delta).clamp(0, 1))
        dist_p = ((box - trapezoid).abs()**p * h).sum().item()
        assert abs(dist_p - 2 * delta / (p + 1)) < 1e-5
print("||1_[0,1) - trapezoid||_p^p = 2 delta / (p + 1) for p = 1, 2, 5 and delta = 0.2, 0.02")

squares of side 1.00000:  measure of the symmetric difference = 1.53937


squares of side 0.50000:  measure of the symmetric difference = 0.53979


squares of side 0.25000:  measure of the symmetric difference = 0.30050
squares of side 0.12500:  measure of the symmetric difference = 0.16281
squares of side 0.06250:  measure of the symmetric difference = 0.05513


squares of side 0.03125:  measure of the symmetric difference = 0.03233


squares of side 0.01562:  measure of the symmetric difference = 0.01713


||1_[0,1) - trapezoid||_p^p = 2 delta / (p + 1) for p = 1, 2, 5 and delta = 0.2, 0.02


## 6. Conditional expectation is a projection (Theorem 19.9, Example 19.10, Figure 19.2, Exercise 19.5)

The die with the blocks $\{1,2\}$, $\{3\}$, $\{4,5,6\}$, exactly.

In [9]:
faces = [1, 2, 3, 4, 5, 6]
P = {z: Fraction(1, 6) for z in faces}
blocks = [{1, 2}, {3}, {4, 5, 6}]
W = {}
for B in blocks:
    avg = sum(z * P[z] for z in B) / sum(P[z] for z in B)
    for z in B:
        W[z] = avg
print("W = E[Z | G]:", {z: str(W[z]) for z in faces})

Ex = lambda fn: sum(fn(z) * P[z] for z in faces)
for B in blocks:                                         # Z - W is orthogonal to each indicator
    assert Ex(lambda z: (z - W[z]) * (1 if z in B else 0)) == 0
var_Z = Ex(lambda z: z**2) - Ex(lambda z: z)**2
residual = Ex(lambda z: (z - W[z])**2)
var_W = Ex(lambda z: W[z]**2) - Ex(lambda z: W[z])**2
print(f"Var(Z) = {var_Z} = E[(Z - W)^2] + Var(W) = {residual} + {var_W}")
assert var_Z == Fraction(35, 12) and residual == Fraction(5, 12) and var_W == Fraction(5, 2)
assert Ex(lambda z: W[z]) == Ex(lambda z: z)             # E[E[Z | G]] = E[Z]

W = E[Z | G]: {1: '3/2', 2: '3/2', 3: '3', 4: '5', 5: '5', 6: '5'}
Var(Z) = 35/12 = E[(Z - W)^2] + Var(W) = 5/12 + 5/2


The same on a finite probability space with random weights, where the
projection is computed as a weighted least-squares problem and compared
with the block averages; then the identity (3) of Theorem 19.9 for random
$\mathcal G$-measurable $V$.

In [10]:
m = 12
p_w = torch.rand(m)
p_w = p_w / p_w.sum()
Zv = torch.randn(m)
label = T([0, 0, 0, 1, 1, 2, 2, 2, 2, 3, 3, 3])          # four blocks
A = torch.nn.functional.one_hot(label).double()          # columns: indicators of the blocks

# minimise sum_i p_i (Z_i - (A c)_i)^2
sq = p_w.sqrt()[:, None]
c = torch.linalg.lstsq(sq * A, sq[:, 0] * Zv).solution
W_ls = A @ c
W_avg = torch.stack([(Zv * p_w)[label == b].sum() / p_w[label == b].sum() for b in range(4)])[label]
print("least squares:  ", [f"{v:+.4f}" for v in c.tolist()])
print("block averages: ", [f"{v:+.4f}" for v in W_avg[[0, 3, 5, 9]].tolist()])
assert torch.allclose(W_ls, W_avg)

Ew = lambda v: (v * p_w).sum()
for _ in range(5):
    V = A @ torch.randn(4)                               # any G-measurable function
    assert abs(Ew((Zv - V)**2) - (Ew((Zv - W_avg)**2) + Ew((W_avg - V)**2))) < 1e-12
    assert abs(Ew(W_avg * V) - Ew(Zv * V)) < 1e-12       # the characterisation (1)
print("Pythagoras and E[W V] = E[Z V] hold for random G-measurable V")

least squares:   ['+0.8696', '+0.6015', '-0.0549', '-0.1458']
block averages:  ['+0.8696', '+0.6015', '-0.0549', '-0.1458']
Pythagoras and E[W V] = E[Z V] hold for random G-measurable V


The block averages drawn in Figure 19.2, for
$Z(\omega) = 0.25 + 0.9\omega + 0.35\sin(3\pi\omega)$ and four equal
blocks.

In [11]:
zfun = lambda t: 0.25 + 0.9 * t + 0.35 * torch.sin(3 * math.pi * t)
averages = [integral(zfun, a, a + 0.25).item() / 0.25 for a in (0.0, 0.25, 0.5, 0.75)]
print("block averages:", [f"{v:.3f}" for v in averages])
assert all(abs(a - b) < 6e-4 for a, b in zip(averages, [0.616, 0.482, 0.707, 1.291]))

block averages: ['0.616', '0.482', '0.707', '1.291']


**The regression function without densities, and variance reduction**
(the key idea and the machine-learning box of §19.3). Samples $(x, y)$ of
the density $x + y$ of Chapter 18 are drawn by rejection (seeded). The
least-squares fit of $y$ on polynomials of $x$, a projection on a
finite-dimensional subspace of functions of $x$, approaches
$m(x) = \frac{3x + 2}{6x + 3}$, and the excess risk of a predictor is its
squared distance to $m$.

In [12]:
N = 400_000
cand = torch.rand(3 * N, 3)
keep = cand[:, 2] * 2 < cand[:, 0] + cand[:, 1]          # accept with probability (x + y) / 2
xs, ys = cand[keep][:N, 0], cand[keep][:N, 1]
assert len(xs) == N

m_true = lambda t: (3 * t + 2) / (6 * t + 3)
features = torch.stack([xs**d for d in range(6)], dim=1)
coef = torch.linalg.lstsq(features, ys).solution
grid = torch.linspace(0, 1, 11)
fit = torch.stack([grid**d for d in range(6)], dim=1) @ coef
print("x      :", [f"{v:.1f}" for v in grid.tolist()])
print("fit    :", [f"{v:.3f}" for v in fit.tolist()])
print("m(x)   :", [f"{v:.3f}" for v in m_true(grid).tolist()])
assert (fit - m_true(grid)).abs().max() < 0.02

risk = lambda pred: ((ys - pred)**2).mean().item()
for name, pred in [("h(x) = x", xs), ("h(x) = 0.6", torch.full_like(xs, 0.6))]:
    excess = risk(pred) - risk(m_true(xs))
    dist2 = ((pred - m_true(xs))**2).mean().item()
    print(f"{name:11s} excess risk = {excess:.5f}   ||h - m||^2 = {dist2:.5f}")
    assert abs(excess - dist2) < 2e-3

# Variance reduction: Z = y^2 and its conditional expectation given x, in closed form.
cond = (xs / 3 + 0.25) / (xs + 0.5)                       # E[y^2 | x] = int y^2 (x + y) dy / (x + 1/2)
print(f"mean of y^2 = {(ys**2).mean():.5f}   mean of E[y^2 | x] = {cond.mean():.5f}   (exact 5/12 = {5 / 12:.5f})")
print(f"variance of y^2 = {(ys**2).var():.5f}   variance of E[y^2 | x] = {cond.var():.5f}")
assert abs((ys**2).mean() - 5 / 12) < 3e-3 and abs(cond.mean() - 5 / 12) < 1e-3
assert cond.var() < 0.05 * (ys**2).var()

x      : ['0.0', '0.1', '0.2', '0.3', '0.4', '0.5', '0.6', '0.7', '0.8', '0.9', '1.0']
fit    : ['0.665', '0.640', '0.620', '0.604', '0.592', '0.582', '0.575', '0.569', '0.565', '0.560', '0.555']
m(x)   : ['0.667', '0.639', '0.619', '0.604', '0.593', '0.583', '0.576', '0.569', '0.564', '0.560', '0.556']
h(x) = x    excess risk = 0.09093   ||h - m||^2 = 0.09089
h(x) = 0.6  excess risk = 0.00097   ||h - m||^2 = 0.00096
mean of y^2 = 0.41679   mean of E[y^2 | x] = 0.41663   (exact 5/12 = 0.41667)
variance of y^2 = 0.09309   variance of E[y^2 | x] = 0.00068


## 7. The Radon–Nikodym theorem (Theorem 19.13, Exercise 19.7)

On a finite set the proof can be followed step by step: the Riesz
representer of $g\mapsto\int g\,d\nu$ in $L^2(\mu + \nu)$ is
$u = \nu/(\mu + \nu)$, with values in $[0, 1]$, and $u/(1 - u)$ is the
density $\nu/\mu$.

In [13]:
k = 9
mu_w = 0.2 + torch.rand(k)
nu_w = torch.rand(k)
nu_w[2] = 0.0                                             # nu may vanish where mu does not
rho_w = mu_w + nu_w

# Riesz representer in L^2(rho): solve <u, e_i>_rho = phi(e_i) for the basis of indicators.
gram = torch.diag(rho_w)                                  # <e_i, e_j>_rho
u = torch.linalg.solve(gram, nu_w)                        # phi(e_i) = nu_i
assert torch.allclose(u, nu_w / rho_w) and (u >= 0).all() and (u < 1).all()
f = u / (1 - u)
assert torch.allclose(f, nu_w / mu_w)
gtest = torch.randn(k)
assert abs((gtest * nu_w).sum() - (gtest * f * mu_w).sum()) < 1e-12
print("u = dnu/d(mu+nu):", [f"{v:.3f}" for v in u.tolist()])
print("f = u/(1-u) = dnu/dmu:", [f"{v:.3f}" for v in f.tolist()])

# Exercise 19.7 (1): mu = Lebesgue on [0, 1], nu = 2x dx.
x, h = midpoints(0.0, 1.0, 1_000_000)
u_x = 2 * x / (1 + 2 * x)
for a, b in [(0.0, 0.3), (0.2, 0.9), (0.5, 1.0)]:
    A_ = ((x >= a) & (x < b)).double()
    nu_A = b**2 - a**2
    assert abs((A_ * u_x * (1 + 2 * x) * h).sum() - nu_A) < 1e-9      # nu(A) = int_A u d(mu + nu)
assert torch.allclose(u_x / (1 - u_x), 2 * x)
print("Exercise 19.7 (1): u = 2x/(1 + 2x) and u/(1 - u) = 2x")

u = dnu/d(mu+nu): ['0.721', '0.492', '0.000', '0.440', '0.314', '0.338', '0.375', '0.350', '0.465']
f = u/(1-u) = dnu/dmu: ['2.586', '0.967', '0.000', '0.785', '0.457', '0.511', '0.600', '0.539', '0.868']


Exercise 19.7 (1): u = 2x/(1 + 2x) and u/(1 - u) = 2x


**A discriminator estimates $u$** (fourth item of the machine-learning box
of §19.5). Samples of $P = N(0, 1)$ are labelled $0$ and samples of
$Q = N(1.5, 1)$ are labelled $1$ (seeded), and a logistic regression with
logit $ax + b$ is fitted. Its output estimates
$u = \frac{dQ}{d(P + Q)}$, so its logit estimates
$\ln\frac{u}{1 - u} = \ln\frac{dQ}{dP} = 1.5x - 1.125$.

In [14]:
M = 400_000
xp, xq = torch.randn(M), 1.5 + torch.randn(M)
data = torch.cat([xp, xq])
target = torch.cat([torch.zeros(M), torch.ones(M)])
params = torch.zeros(2, requires_grad=True)
opt = torch.optim.LBFGS([params], lr=1.0, max_iter=200, tolerance_grad=1e-12,
                        tolerance_change=1e-14, line_search_fn="strong_wolfe")


def closure():
    opt.zero_grad()
    loss = torch.nn.functional.binary_cross_entropy_with_logits(params[0] * data + params[1], target)
    loss.backward()
    return loss


opt.step(closure)
a_hat, b_hat = params.tolist()
print(f"fitted logit: {a_hat:.4f} x + ({b_hat:.4f})    log likelihood ratio: 1.5 x - 1.125")
assert abs(a_hat - 1.5) < 0.02 and abs(b_hat + 1.125) < 0.02

fitted logit: 1.5011 x + (-1.1257)    log likelihood ratio: 1.5 x - 1.125


## 8. The Kullback–Leibler divergence (Definition 19.15, Theorem 19.16, Exercises 19.7 and 19.8)

On a finite set: non-negative, zero only for equal distributions, infinite
when the first measure charges a point that the second does not.

In [15]:
def kl(q, p):
    # D(Q || P) for weights on a finite set, with 0 ln 0 = 0.
    if any(qi > 0 and pi == 0 for qi, pi in zip(q, p)):
        return math.inf
    return sum(float(qi) * math.log(qi / pi) for qi, pi in zip(q, p) if qi > 0)


P3 = [Fraction(1, 2), Fraction(1, 2), Fraction(0)]
Q3 = [Fraction(1, 4), Fraction(1, 4), Fraction(1, 2)]
print(f"D(P || Q) = {kl(P3, Q3):.6f} (ln 2 = {math.log(2):.6f})    D(Q || P) = {kl(Q3, P3)}")
assert abs(kl(P3, Q3) - math.log(2)) < 1e-12 and kl(Q3, P3) == math.inf
assert kl(P3, P3) == 0

smallest = math.inf
for _ in range(2000):
    a_, b_ = torch.rand(6), torch.rand(6)
    a_, b_ = (a_ / a_.sum()).tolist(), (b_ / b_.sum()).tolist()
    smallest = min(smallest, kl(a_, b_))
print(f"smallest divergence among 2000 random pairs: {smallest:.5f} > 0")
assert smallest > 0

D(P || Q) = 0.693147 (ln 2 = 0.693147)    D(Q || P) = inf
smallest divergence among 2000 random pairs: 0.00414 > 0


For two Gaussian laws with the same variance,
$D(Q\,\|\,P) = (m_1 - m_2)^2/(2s^2)$, by quadrature of
$\int q\ln\frac qp$. The value does not change under $x\mapsto ax + b$,
while $-\int q\ln q$ changes by $\ln|a|$.

In [16]:
def gauss(mean, s):
    return lambda t: torch.exp(-(t - mean)**2 / (2 * s**2)) / (s * math.sqrt(2 * math.pi))


def kl_density(q, p, lo, hi):
    t, h = midpoints(lo, hi, 2_000_000)
    return (q(t) * (torch.log(q(t)) - torch.log(p(t))) * h).sum().item()


def diff_entropy(q, lo, hi):
    t, h = midpoints(lo, hi, 2_000_000)
    return -(q(t) * torch.log(q(t)) * h).sum().item()


m1, m2, s = 0.3, 1.8, 0.7
d = kl_density(gauss(m2, s), gauss(m1, s), -8.0, 10.0)
print(f"D(Q || P) = {d:.8f}   (m1 - m2)^2 / (2 s^2) = {(m1 - m2)**2 / (2 * s**2):.8f}")
assert abs(d - (m1 - m2)**2 / (2 * s**2)) < 1e-8

ent = diff_entropy(gauss(m2, s), -8.0, 10.0)
assert abs(ent - 0.5 * math.log(2 * math.pi * math.e * s**2)) < 1e-8
for a_, b_ in [(3.0, -1.0), (-0.2, 4.0)]:
    lo, hi = sorted((a_ * -8.0 + b_, a_ * 10.0 + b_))
    d_new = kl_density(gauss(a_ * m2 + b_, abs(a_) * s), gauss(a_ * m1 + b_, abs(a_) * s), lo, hi)
    ent_new = diff_entropy(gauss(a_ * m2 + b_, abs(a_) * s), lo, hi)
    print(f"x -> {a_} x + {b_}:  D = {d_new:.8f}   entropy changes by {ent_new - ent:+.6f}"
          f"   ln|a| = {math.log(abs(a_)):+.6f}")
    assert abs(d_new - d) < 1e-7 and abs(ent_new - ent - math.log(abs(a_))) < 1e-7

s_neg = 0.2
print(f"s = {s_neg}: differential entropy = {0.5 * math.log(2 * math.pi * math.e * s_neg**2):+.4f} < 0;"
      f"  threshold (2 pi e)^(-1/2) = {(2 * math.pi * math.e)**-0.5:.4f}")
assert 0.5 * math.log(2 * math.pi * math.e * s_neg**2) < 0 < 0.5 * math.log(2 * math.pi * math.e * 0.25**2)

D(Q || P) = 2.29591837   (m1 - m2)^2 / (2 s^2) = 2.29591837
x -> 3.0 x + -1.0:  D = 2.29591837   entropy changes by +1.098612   ln|a| = +1.098612


x -> -0.2 x + 4.0:  D = 2.29591837   entropy changes by -1.609438   ln|a| = -1.609438
s = 0.2: differential entropy = -0.1905 < 0;  threshold (2 pi e)^(-1/2) = 0.2420


The divergence needs no density with respect to Lebesgue measure. Cutting
the line into cells and comparing the two probabilities of each cell gives
a divergence between two discrete laws; it increases to the value above as
the cells shrink.

In [17]:
cdf = lambda t, mean, sd: 0.5 * (1 + torch.erf((t - mean) / (sd * math.sqrt(2))))
previous = 0.0
for cells_ in (4, 16, 64, 256, 4096):
    edges = torch.linspace(-8.0, 10.0, cells_ + 1)
    qb = cdf(edges[1:], m2, s) - cdf(edges[:-1], m2, s)
    pb = cdf(edges[1:], m1, s) - cdf(edges[:-1], m1, s)
    ok = (qb > 0) & (pb > 0)
    binned = (qb[ok] * torch.log(qb[ok] / pb[ok])).sum().item()
    print(f"{cells_:5d} cells:  divergence of the binned laws = {binned:.6f}")
    assert previous <= binned + 1e-12 <= d + 1e-9
    previous = binned
assert d - previous < 1e-4

    4 cells:  divergence of the binned laws = 1.250131
   16 cells:  divergence of the binned laws = 1.910646
   64 cells:  divergence of the binned laws = 2.265621
  256 cells:  divergence of the binned laws = 2.293990
 4096 cells:  divergence of the binned laws = 2.295911


## 9. Likelihood ratios as weights (Figure 19.3, Exercise 19.9)

$P = N(0, 1)$, $Q = N(\theta, 1)$, $w = e^{\theta x - \theta^2/2}$. By
quadrature $\mathbb E_P[w] = 1$, $\mathbb E_P[xw] = \theta$ and
$\mathbb E_P[w^2] = e^{\theta^2}$.

In [18]:
x, h = midpoints(-30.0, 30.0, 4_000_000)
p0 = torch.exp(-x**2 / 2) / math.sqrt(2 * math.pi) * h
for theta in (0.5, 1.0, 1.5, 3.0):
    wgt = torch.exp(theta * x - theta**2 / 2)
    mass, mean, second = (wgt * p0).sum().item(), (x * wgt * p0).sum().item(), (wgt**2 * p0).sum().item()
    needed = 100 * (math.exp(theta**2) - 1)
    print(f"theta = {theta}:  E[w] = {mass:.8f}   E[x w] = {mean:.8f}   E[w^2] = {second:.4f}"
          f" = e^(theta^2) = {math.exp(theta**2):.4f}   samples for sd 0.1: {needed:,.0f}")
    assert abs(mass - 1) < 1e-8 and abs(mean - theta) < 1e-8
    assert abs(second / math.exp(theta**2) - 1) < 1e-8
assert round(100 * (math.e - 1)) == 172 and round(100 * (math.exp(9) - 1)) == 810_208

# the ratio of Figure 19.3 equals 1 where the two densities cross
assert abs(math.exp(1.5 * 0.75 - 1.125) - 1) < 1e-15

theta = 0.5:  E[w] = 1.00000000   E[x w] = 0.50000000   E[w^2] = 1.2840 = e^(theta^2) = 1.2840   samples for sd 0.1: 28
theta = 1.0:  E[w] = 1.00000000   E[x w] = 1.00000000   E[w^2] = 2.7183 = e^(theta^2) = 2.7183   samples for sd 0.1: 172
theta = 1.5:  E[w] = 1.00000000   E[x w] = 1.50000000   E[w^2] = 9.4877 = e^(theta^2) = 9.4877   samples for sd 0.1: 849
theta = 3.0:  E[w] = 1.00000000   E[x w] = 3.00000000   E[w^2] = 8103.0839 = e^(theta^2) = 8103.0839   samples for sd 0.1: 810,208


With samples (seeded): $172$ weights at $\theta = 1$ average to $1$ with
standard deviation about $0.1$; and the weighted average of $x$ estimates
the mean of $Q$ from samples of $P$.

In [19]:
theta = 1.0
reps = 20_000
xs_ = torch.randn(reps, 172)
ws = torch.exp(theta * xs_ - theta**2 / 2)
avg_w = ws.mean(dim=1)
est_mean = (xs_ * ws).mean(dim=1)
print(f"average of 172 weights: mean = {avg_w.mean():.4f}   standard deviation = {avg_w.std():.4f}")
print(f"importance estimate of E_Q[x]: mean = {est_mean.mean():.4f}   (theta = {theta})")
assert abs(avg_w.mean() - 1) < 5e-3 and abs(avg_w.std() - 0.1) < 0.01
assert abs(est_mean.mean() - theta) < 0.01

average of 172 weights: mean = 1.0011   standard deviation = 0.1008
importance estimate of E_Q[x]: mean = 1.0033   (theta = 1.0)


## 10. The evidence lower bound (last item of the machine-learning box of §19.5)

A latent variable with three values and a fixed observation $x$. For any
distribution $r$ on the latent values,
$\ln p(x) - \mathbb E_r[\ln\frac{p(x, z)}{r(z)}] = D(r\,\|\,p(\cdot\mid x))\geq0$,
with equality for the posterior.

In [20]:
joint = T([0.10, 0.02, 0.28])                          # p(x, z) for z = 0, 1, 2 and the observed x
evidence = joint.sum()
posterior = joint / evidence


def elbo(r):
    return (r * (torch.log(joint) - torch.log(r))).sum()


for _ in range(5):
    r = torch.rand(3)
    r = r / r.sum()
    gap = torch.log(evidence) - elbo(r)
    div = (r * torch.log(r / posterior)).sum()
    print(f"ln p(x) = {torch.log(evidence):.5f}   ELBO = {elbo(r):.5f}   gap = {gap:.5f}   D(r || posterior) = {div:.5f}")
    assert gap > 0 and abs(gap - div) < 1e-12
assert abs(torch.log(evidence) - elbo(posterior)) < 1e-12
print("equality for r = posterior")

ln p(x) = -0.91629   ELBO = -1.49421   gap = 0.57792   D(r || posterior) = 0.57792
ln p(x) = -0.91629   ELBO = -1.69063   gap = 0.77434   D(r || posterior) = 0.77434
ln p(x) = -0.91629   ELBO = -1.36655   gap = 0.45026   D(r || posterior) = 0.45026
ln p(x) = -0.91629   ELBO = -2.29586   gap = 1.37957   D(r || posterior) = 1.37957
ln p(x) = -0.91629   ELBO = -1.66224   gap = 0.74595   D(r || posterior) = 0.74595
equality for r = posterior


## 11. Convolution in $L^p$ (Theorems 19.17 and 19.19, Lemma 19.18, Figure 19.4, Exercise 19.10)

Young's inequality for convolutions on $\mathbb Z$ with the counting
measure, where it is exact for finite sequences:
$\|f*g\|_p\leq\|f\|_1\|g\|_p$. For non-negative sequences and $p = 1$ it
is an equality.

In [21]:
def conv_full(f, g):
    # (f * g)_k = sum_j f_j g_{k-j}, all k
    return torch.nn.functional.conv1d(g.view(1, 1, -1), f.flip(0).view(1, 1, -1),
                                      padding=len(f) - 1).view(-1)


ones = lambda v: torch.ones(len(v))
for p in (1.0, 1.5, 2.0, 4.0, math.inf):
    worst = 0.0
    for _ in range(200):
        f, g = torch.randn(9), torch.randn(200)
        c = conv_full(f, g)
        ratio = norm(c, ones(c), p) / (norm(f, ones(f), 1.0) * norm(g, ones(g), p))
        worst = max(worst, ratio.item())
    print(f"p = {p:4}:  max of ||f * g||_p / (||f||_1 ||g||_p) = {worst:.4f}")
    assert worst <= 1 + 1e-12

f, g = torch.rand(9), torch.rand(200)
c = conv_full(f, g)
assert abs(c.sum() - f.sum() * g.sum()) < 1e-9             # equality for p = 1, f, g >= 0
assert torch.allclose(conv_full(f, g), conv_full(g, f))    # f * g = g * f

p =  1.0:  max of ||f * g||_p / (||f||_1 ||g||_p) = 0.5248
p =  1.5:  max of ||f * g||_p / (||f||_1 ||g||_p) = 0.5370
p =  2.0:  max of ||f * g||_p / (||f||_1 ||g||_p) = 0.5624
p =  4.0:  max of ||f * g||_p / (||f||_1 ||g||_p) = 0.5593
p =  inf:  max of ||f * g||_p / (||f||_1 ||g||_p) = 0.6956


A convolution layer with one filter is Lipschitz in every $\ell^p$ with
constant at most the sum of the absolute values of its weights (second
item of the machine-learning box of §19.6). Checked on random pairs of
images with `conv2d`, which computes the same sums up to a reflection of
the filter.

In [22]:
filt = torch.randn(1, 1, 3, 3)
l1 = filt.abs().sum().item()
for p in (1.0, 2.0, 3.0, math.inf):
    worst = 0.0
    for _ in range(100):
        a_img, b_img = torch.randn(1, 1, 32, 32), torch.randn(1, 1, 32, 32)
        out = torch.nn.functional.conv2d(a_img, filt, padding=2) - torch.nn.functional.conv2d(b_img, filt, padding=2)
        diff = (a_img - b_img).flatten()
        ratio = norm(out.flatten(), ones(out.flatten()), p) / norm(diff, ones(diff), p)
        worst = max(worst, ratio.item())
    print(f"p = {p:4}:  largest ratio of output distance to input distance = {worst:.4f}"
          f"   ||filter||_1 = {l1:.4f}")
    assert worst <= l1 + 1e-12

p =  1.0:  largest ratio of output distance to input distance = 2.9803   ||filter||_1 = 6.4608
p =  2.0:  largest ratio of output distance to input distance = 2.8281   ||filter||_1 = 6.4608
p =  3.0:  largest ratio of output distance to input distance = 2.7944   ||filter||_1 = 6.4608
p =  inf:  largest ratio of output distance to input distance = 3.5814   ||filter||_1 = 6.4608


Smoothing the indicator of $[0, 1]$ by a moving average of half-width
$\sigma$ (Figure 19.4, Exercise 19.10). The convolution is computed on a
grid and compared with the trapezoid; then
$\|k_\sigma*g - g\|_1 = \sigma$, $\|k_\sigma*g - g\|_2^2 = \sigma/3$,
$\|k_\sigma*g - g\|_\infty = \frac12$ and
$\|k_\sigma*g\|_2^2 = 1 - \frac{2\sigma}3$.

In [23]:
hh = 1e-4
xg = torch.arange(-1.0, 2.0, hh) + hh / 2
g_ind = ((xg >= 0) & (xg <= 1)).double()
for sigma in (0.4, 0.2, 0.05):
    half = int(round(sigma / hh))
    kernel = torch.full((2 * half + 1,), 1 / (2 * sigma))        # k_sigma on the grid, total weight 1
    kernel[0] = kernel[-1] = 1 / (4 * sigma)
    smooth = torch.nn.functional.conv1d(g_ind.view(1, 1, -1), kernel.view(1, 1, -1),
                                        padding=half).view(-1) * hh
    trapezoid = torch.minimum(((xg + sigma) / (2 * sigma)).clamp(0, 1),
                              ((1 + sigma - xg) / (2 * sigma)).clamp(0, 1))
    assert (smooth - trapezoid).abs().max() < 2e-3
    d = trapezoid - g_ind
    d1, d2, dinf = (d.abs() * hh).sum().item(), (d**2 * hh).sum().item(), d.abs().max().item()
    n2 = (trapezoid**2 * hh).sum().item()
    print(f"sigma = {sigma}:  L1 distance = {d1:.5f}   L2 distance squared = {d2:.5f} (sigma/3 = {sigma / 3:.5f})"
          f"   sup distance = {dinf:.4f}   ||k*g||_2^2 = {n2:.5f}")
    assert abs(d1 - sigma) < 1e-4 and abs(d2 - sigma / 3) < 1e-4 and abs(dinf - 0.5) < 2e-3
    assert abs(n2 - (1 - 2 * sigma / 3)) < 1e-4 and n2 <= 1

# Lemma 19.18 on the same function: ||tau_v g - g||_p^p = 2|v| for p < infinity, and 1 for the sup norm.
for v in (0.3, 0.03, 0.003):
    shifted = ((xg - v >= 0) & (xg - v <= 1)).double()
    dp = ((shifted - g_ind).abs() * hh).sum().item()
    print(f"v = {v}:  ||tau_v g - g||_p^p = {dp:.5f} = 2|v|   sup norm = {(shifted - g_ind).abs().max():.0f}")
    assert abs(dp - 2 * v) < 3e-4 and (shifted - g_ind).abs().max() == 1

sigma = 0.4:  L1 distance = 0.40000   L2 distance squared = 0.13333 (sigma/3 = 0.13333)   sup distance = 0.4999   ||k*g||_2^2 = 0.73333


sigma = 0.2:  L1 distance = 0.20000   L2 distance squared = 0.06667 (sigma/3 = 0.06667)   sup distance = 0.4999   ||k*g||_2^2 = 0.86667
sigma = 0.05:  L1 distance = 0.05000   L2 distance squared = 0.01667 (sigma/3 = 0.01667)   sup distance = 0.4995   ||k*g||_2^2 = 0.96667
v = 0.3:  ||tau_v g - g||_p^p = 0.60000 = 2|v|   sup norm = 1
v = 0.03:  ||tau_v g - g||_p^p = 0.06000 = 2|v|   sup norm = 1
v = 0.003:  ||tau_v g - g||_p^p = 0.00600 = 2|v|   sup norm = 1


Theorem 19.19 for an unbounded function and a Gaussian kernel:
$g(x) = x^{-1/4}1_{(0, 1)}$, which is in $L^1$ and $L^2$. The distances
$\|\varphi_\sigma*g - g\|_p$ decrease with $\sigma$, slowly for $p = 2$
because of the singularity at $0$, and
$\|\varphi_\sigma*g\|_p\leq\|g\|_p$ (the first item of the same box).

In [24]:
hh = 2e-4
xg = torch.arange(-2.0, 3.0, hh) + hh / 2
g_sing = torch.where((xg > 0) & (xg < 1), xg.clamp(min=hh / 2)**-0.25, torch.zeros_like(xg))
prev = {1: math.inf, 2: math.inf}
for sigma in (0.2, 0.05, 0.0125, 0.003):
    half = int(round(6 * sigma / hh))
    t = torch.arange(-half, half + 1) * hh
    kernel = torch.exp(-t**2 / (2 * sigma**2))
    kernel = kernel / (kernel.sum() * hh)                         # total weight 1 on the grid
    smooth = torch.nn.functional.conv1d(g_sing.view(1, 1, -1), kernel.view(1, 1, -1),
                                        padding=half).view(-1) * hh
    line = f"sigma = {sigma:6.4f}: "
    for p in (1, 2):
        dist = (((smooth - g_sing).abs()**p * hh).sum())**(1 / p)
        n_s, n_g = ((smooth.abs()**p * hh).sum())**(1 / p), ((g_sing.abs()**p * hh).sum())**(1 / p)
        line += f"  ||k*g - g||_{p} = {dist:.5f}   ||k*g||_{p} = {n_s:.5f} <= ||g||_{p} = {n_g:.5f} "
        assert dist < prev[p] and n_s <= n_g + 1e-9
        prev[p] = dist.item()
    print(line)
assert prev[1] < 0.02 and prev[2] < 0.2

sigma = 0.2000:   ||k*g - g||_1 = 0.49522   ||k*g||_1 = 1.33307 <= ||g||_1 = 1.33307   ||k*g - g||_2 = 0.62835   ||k*g||_2 = 1.16269 <= ||g||_2 = 1.41119 


sigma = 0.0500:   ||k*g - g||_1 = 0.16436   ||k*g||_1 = 1.33307 <= ||g||_1 = 1.33307   ||k*g - g||_2 = 0.42713   ||k*g||_2 = 1.30716 <= ||g||_2 = 1.41119 
sigma = 0.0125:   ||k*g - g||_1 = 0.05394   ||k*g||_1 = 1.33307 <= ||g||_1 = 1.33307   ||k*g - g||_2 = 0.29101   ||k*g||_2 = 1.36421 <= ||g||_2 = 1.41119 
sigma = 0.0030:   ||k*g - g||_1 = 0.01728   ||k*g||_1 = 1.33307 <= ||g||_1 = 1.33307   ||k*g - g||_2 = 0.19319   ||k*g||_2 = 1.39029 <= ||g||_2 = 1.41119 


## Summary

* Jensen, Young, Hölder and Minkowski hold on every example tried; Hölder is
  an equality for $g = \operatorname{sign}(f)|f|^{q-1}/\|f\|_q^{q-1}$.
* $\|f\|_p$ increases with $p$ on a probability space and decreases for the
  counting measure; $\|x\|_p = (p+1)^{-1/p}\to1$ on $[0, 1]$.
* The ramps converge in $L^1$ to a step function at distance $1/(2k)$; the
  sliding indicators have norms $2^{-n/p}\to0$ and converge at no point.
* Conditional expectation given a partition is the block average, equals a
  weighted least-squares projection, and satisfies Pythagoras; for the die,
  $\frac{35}{12} = \frac5{12} + \frac52$.
* On a finite set $u = \nu/(\mu + \nu)$ and $u/(1 - u) = \nu/\mu$; a logistic
  discriminator between $N(0,1)$ and $N(1.5,1)$ recovers the log-ratio
  $1.5x - 1.125$.
* $D(P\|Q) = \ln 2$ and $D(Q\|P) = \infty$ in Exercise 19.7; for Gaussians
  $D = (m_1 - m_2)^2/(2s^2)$, invariant under affine maps, while the
  differential entropy shifts by $\ln|a|$.
* Importance weights between $N(0,1)$ and $N(\theta,1)$ have second moment
  $e^{\theta^2}$: $172$ samples for a standard deviation $0.1$ at
  $\theta = 1$, $810\,208$ at $\theta = 3$.
* The gap of the evidence lower bound is the divergence to the posterior.
* $\|f*g\|_p\leq\|f\|_1\|g\|_p$ on sequences and for a `conv2d` layer; the
  moving average of $1_{[0,1]}$ is at $L^1$ distance $\sigma$ and at sup
  distance $\frac12$ from it.